# BP4 Executive Rollup Report — Customer Journey Analytics
**Customer360 Navigator Enterprise Suite — Customer Journey Analytics**

## Purpose
Produces BP4's single, comprehensive executive-rollup deliverable ("Gate 7"): a world-class
interactive HTML dashboard (animated KPIs, a tier slicer + search filter + sortable/paginated
table over the real issue-cluster population, dynamic Plotly charts, SMART suggestions, and the
live "Recommended for Production" status), a Word report, an Excel workbook, and a PowerPoint
deck — all built exclusively from BP4 Gates 1-6's real, already-confirmed artifacts. This mirrors
the standing Gate 7 pattern established for BP1/BP2/BP3, adapted for BP4's real shape.

## Why this Gate 7 looks different from BP1/BP2/BP3's
BP4 trains no model and has no supervised target — the classifier-specific content every prior
BP3 Gate 7 carried (PR-AUC/ROC-AUC, a confusion matrix, SHAP, a disparate-impact check) simply
does not exist for BP4. Three deliberate structural adaptations follow:
- **Gate 3's real content is an aggregation-pipeline benchmark**, not a model benchmark — 5
  candidate implementations (pandas_groupby, polars_eager, polars_lazy, polars_lazy_streaming,
  duckdb_sql) raced on real wall-clock speed among CORRECT candidates only.
- **Gate 4's real content is four bootstrap-CI statistics** (mean response lag,
  in-scope-vs-out-of-scope lag difference, recurring-cluster rate, mean cluster size), not a
  single held-out metric.
- **The "Recommended for Production" status can only ever resolve to Tier 1 (RECOMMENDED FOR
  PRODUCTION) or Tier 3 (NOT RECOMMENDED) for BP4** — Tier 2 (CONDITIONAL - GOVERNANCE REVIEW
  REQUIRED) has no possible trigger, because Gate 4's real
  `ecoa_disparate_impact_applicability` is `NOT_APPLICABLE` (ECOA/Reg B does not map to BP4 per
  Master Plan Section 9). This is stated explicitly in the computed reason string on every
  deliverable, never silently omitted.

## What this notebook does, concretely
1. **Reads BP4 Gates 1-6's real artifacts live** via the new shared module
   `src/reporting/bp4_rollup_helpers.py` (sibling to `bp1/bp2/bp3_rollup_helpers.py`, same
   `PALETTE`/`CATEGORICAL_SEQUENCE` design-system identity reused verbatim — HYPER): the config
   YAML, `policy.json`, `gate2_feature_lineage.csv`, `gate3_benchmark_results.csv`,
   `gate5_decision_layer_summary.json`, `gate5_cluster_decision_report.csv`, and
   `gate6_governance_summary.json`.
2. **Computes the live "Recommended for Production" status** (`compute_production_recommendation`)
   from real structural-integrity signals only — never asserted in prose.
3. **Live-aggregates a real monthly complaint-volume trend** from the Gate 2 Gold table
   (`data/processed/cfpb_issue_cluster_monthly_gold.parquet`) via Polars — a genuinely real,
   non-fabricated addition, computed fresh by this notebook, never hardcoded.
4. **Renders every static figure once** (WARP) — reused as raw PNG bytes across the DOCX and
   PPTX exports.
5. **Builds the interactive HTML dashboard.** Every real HIGH-tier and MEDIUM-tier issue cluster
   (the two tiers carrying a triggered review flag) is embedded as a compact array-of-arrays JSON
   payload, giving the dashboard's tier slicer, search filter, column sorting, and pagination
   something substantial and fully real to explore client-side, without embedding all 37,160
   real clusters (which would needlessly bloat the file — LOW and NONE tier clusters carry no
   triggered review flag and are already fully summarized in the tier-rollup chart and table).
6. **Exports the Word report, Excel workbook, and PowerPoint deck** from the identical KPI bundle
   (HYPER: computed once, reused by every export).
7. **Writes an executive-rollup manifest** (audit trail) and runs structural integrity checks that
   `raise AssertionError`, never silently pass.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: every number in every deliverable is read live from
  Gates 1-6's own already-recorded real artifacts, or computed live from them (the monthly-volume
  trend, the tier-filtered table) — never a remembered or assumed figure.
- **No financial-impact, illustrative, or assumption-based content anywhere** — permanently banned
  project-wide.
- **HYPER**: `PALETTE`/`CATEGORICAL_SEQUENCE` reused verbatim from BP1/BP2/BP3's own rollup
  helpers; matplotlib figures rendered once and reused as PNG bytes across DOCX/PPTX (WARP).
- **Lighter verification for Gate 7** (the standing rule confirmed verbatim from BP2's and BP3's
  own delivered Gate 7 notebooks): "No execution-based verification of this notebook's own code
  was performed by Claude — not even against synthetic fixtures" for the *real run itself*. Before
  delivery, Claude did run `src/reporting/bp4_rollup_helpers.py` and this notebook's own logic
  against a small sandbox mirroring the real artifact schemas (never the real data), plus a
  headless-browser functional check of the HTML dashboard's interactivity (tier filter, search,
  sort, pagination, and the CDN-unavailable resilience fallback) — a pre-delivery bug-catching
  step, never presented as "real-run confirmed."

## Outputs (idempotent overwrite-in-place, under `reports/bp4_customer_journey_analytics/executive_rollup/`)
- `bp4_executive_rollup_dashboard.html`
- `bp4_executive_rollup_report.docx`
- `bp4_executive_rollup_workbook.xlsx`
- `bp4_executive_rollup_deck.pptx`
- `notebooks/bp4_customer_journey_analytics/artifacts/executive_rollup_manifest.json`

## Prerequisites
BP4 Gates 1-6 must all have been real-run at least once. This notebook raises a clear
`FileNotFoundError` naming whichever real artifact is missing.

## If a structural check below fails
It raises `AssertionError` naming the failing check. Do not silence it.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp4_customer_journey_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp4_customer_journey_analytics" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp4_customer_journey_analytics"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ROLLUP_DIR = REPORTS_DIR / "executive_rollup"
ROLLUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")
print(f"[OK] Executive-rollup output folder ready: {ROLLUP_DIR.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Load the shared reporting helper module (HYPER) and all real Gate 1-6 artifacts
# ============================================================
import reporting.bp4_rollup_helpers as rollup  # noqa: E402

_missing_reporting_deps = []
for _mod in ("docx", "openpyxl", "pptx"):
    try:
        __import__(_mod)
    except ImportError:
        _missing_reporting_deps.append(_mod)
if _missing_reporting_deps:
    raise ImportError(
        "Executive-rollup report requires the following packages, missing from this environment: "
        f"{_missing_reporting_deps}. Install them (pip install python-docx openpyxl python-pptx) in "
        "the same environment this notebook runs in, then re-run."
    )
print("[OK] docx / openpyxl / pptx reporting dependencies available.")

BUNDLE = rollup.load_all_gate_artifacts(PROJECT_ROOT)
print(f"[OK] Loaded real Gate 1-6 artifacts. Champion aggregation pipeline: {BUNDLE['champion_pipeline']}")
print(f"[OK] Real Gate 5 cluster-decision export: {len(BUNDLE['gate5_cluster_df']):,} clusters.")

# ============================================================
# SECTION 4: KPI bundle (incl. the new live Recommended-for-Production status), Gate 1 + Gate 6
# real-output detail, SMART suggestions, tier-rollup table - all real, computed live from Gates
# 1-6's own artifacts (HYPER: one computation each, reused by every export). No financial-impact /
# assumption-based section exists anywhere in this notebook, per standing instruction.
# ============================================================
KPIS = rollup.build_kpi_bundle(BUNDLE)
GATE1 = rollup.build_gate1_summary(BUNDLE)
GATE6_DETAIL = rollup.build_gate6_governance_detail(BUNDLE)
SUGGESTIONS = rollup.build_smart_suggestions(BUNDLE)
TIER_DF = rollup.tier_rollup_dataframe(BUNDLE)
print(
    f"[OK] KPI bundle assembled ({len(KPIS)} keys). Recommended for Production: "
    f"{KPIS['production_recommendation']['tier']}. Gate 1 policy detail ({len(GATE1)} fields), "
    f"Gate 6 governance detail ({len(GATE6_DETAIL)} fields) assembled. {len(SUGGESTIONS)} SMART "
    "suggestions generated (all data-grounded, never freeform GenAI text). Tier rollup: "
    f"{len(TIER_DF)} tiers."
)

# ============================================================
# SECTION 5: Live-aggregate the real monthly complaint-volume trend from the Gate 2 Gold table
# (a genuinely real, non-fabricated addition - computed fresh by this notebook, never hardcoded).
# ============================================================
import polars as pl  # noqa: E402

_monthly_gold_path = PROJECT_ROOT / "data" / "processed" / "cfpb_issue_cluster_monthly_gold.parquet"
if not _monthly_gold_path.exists():
    raise FileNotFoundError(
        f"Real Gate 2 Gold table not found at {_monthly_gold_path} - run BP4 Gate 2 for real first."
    )
_monthly_totals_df = (
    pl.scan_parquet(_monthly_gold_path)
    .group_by("complaint_month")
    .agg(pl.col("n_complaints_month").sum().alias("n_complaints_month"))
    .sort("complaint_month")
    .collect()
    .to_pandas()
)
print(f"[OK] Real monthly complaint-volume trend live-aggregated: {len(_monthly_totals_df):,} months.")

# ============================================================
# SECTION 6: Render every static figure ONCE (WARP) - reused as raw PNG bytes across the DOCX and
# PPTX exports below, and never touched as a live pyplot Figure object more than once.
# ============================================================
FIGURES = {
    "benchmark": rollup.fig_benchmark_bar(BUNDLE["gate3_benchmark_df"], KPIS["champion_pipeline"]),
    "tier_rollup": rollup.fig_tier_rollup_bar(TIER_DF),
    "mean_lag_ci": rollup.fig_ci_bar(
        KPIS["mean_response_lag_days"]["point_estimate"],
        KPIS["mean_response_lag_days"]["ci_95_low"],
        KPIS["mean_response_lag_days"]["ci_95_high"],
        "Mean response lag (days)",
    ),
    "recurring_rate_ci": rollup.fig_ci_bar(
        KPIS["recurring_cluster_rate"]["point_estimate"],
        KPIS["recurring_cluster_rate"]["ci_95_low"],
        KPIS["recurring_cluster_rate"]["ci_95_high"],
        "Recurring-cluster rate",
    ),
    "monthly_trend": rollup.fig_monthly_volume_trend(_monthly_totals_df),
}
for _name, _png_bytes in FIGURES.items():
    print(f"[OK] Figure rendered: {_name} ({len(_png_bytes):,} bytes PNG)")

# ============================================================
# SECTION 7: Interactive HTML dashboard (Plotly.js via CDN, a live tier slicer + search filter +
# sortable/paginated table over the real issue-cluster population - no new Python dependency). All
# data embedded as one JSON payload; template placeholders substituted via plain string .replace(),
# never f-strings, to avoid brace-escaping conflicts with the template's own CSS/JS. Every real
# HIGH-tier and MEDIUM-tier issue cluster (the two tiers carrying a triggered review flag) is
# embedded as a compact array-of-arrays JSON payload - LOW/NONE tier clusters carry no triggered
# review flag and stay summarized in the tier-rollup chart/table only, keeping the file size
# reasonable while never truncating the real HIGH/MEDIUM population itself.
# ============================================================
import json  # noqa: E402

import numpy as np  # noqa: E402


def _json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")


_CLUSTER_COLS = [
    "Company",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "n_complaints_total",
    "avg_response_lag_days",
    "banking77_coverage_fraction",
    "review_priority_tier",
    "review_priority_score",
]
_embed_clusters_df = BUNDLE["gate5_cluster_df"]
_embed_clusters_df = _embed_clusters_df[_embed_clusters_df["review_priority_tier"].isin(["HIGH", "MEDIUM"])][
    _CLUSTER_COLS
]

ROLLUP_DATA = {
    "palette": rollup.PALETTE,
    "categorical_sequence": rollup.CATEGORICAL_SEQUENCE,
    "kpis": KPIS,
    "gate1": GATE1,
    "gate6_detail": GATE6_DETAIL,
    "tier_rollup": TIER_DF.to_dict("records"),
    "benchmark": BUNDLE["gate3_benchmark_df"][
        ["candidate", "status", "min_seconds", "mean_seconds", "is_champion"]
    ].to_dict("records"),
    "monthly_trend": [
        {"month": r["complaint_month"], "n_complaints": int(r["n_complaints_month"])}
        for _, r in _monthly_totals_df.iterrows()
    ],
    "clusters": _embed_clusters_df.values.tolist(),
    "suggestions": SUGGESTIONS,
}

_data_json = json.dumps(ROLLUP_DATA, default=_json_default)
_data_json_safe = _data_json.replace("</", "<\\/")  # never let embedded text close the script tag early
print(f"[OK] Rollup data JSON payload assembled: {len(_data_json_safe):,} bytes.")
print(f"[OK] Real HIGH+MEDIUM tier clusters embedded: {len(_embed_clusters_df):,}.")

import re as _re  # noqa: E402

# Real captured pytest stdout (pytest_summary_line) can carry ANSI color-escape control codes on
# some terminals/pytest.ini configs (a real, previously-found issue in this project, guarded here
# pre-emptively for the HTML footer, matching bp1/bp2/bp3's own XLSX writers' guard).
_ANSI_ESCAPE_RE = _re.compile(r"\x1b\[[0-9;]*m")

_pytest_counts = KPIS["pytest_counts"]
_pytest_summary_raw = BUNDLE["gate6_summary"].get(
    "pytest_summary_line", f"{_pytest_counts['passed']} passed / {_pytest_counts['failed']} failed"
)
_pytest_summary = _ANSI_ESCAPE_RE.sub("", _pytest_summary_raw).strip()
_syntax_n_pass = BUNDLE["gate6_summary"]["notebook_syntax_check_n_passed"]
_syntax_n_fail = BUNDLE["gate6_summary"]["notebook_syntax_check_n_failed"]
_syntax_summary = f"{_syntax_n_pass}/{_syntax_n_pass + _syntax_n_fail} passed"

TEMPLATE_PATH = PROJECT_ROOT / "src" / "reporting" / "templates" / "bp4_dashboard_template.html"
_html_text = TEMPLATE_PATH.read_text(encoding="utf-8")
_html_text = _html_text.replace("__GENERATED_AT__", KPIS["generated_at_utc"])
_html_text = _html_text.replace("__CHAMPION_PIPELINE__", KPIS["champion_pipeline"])
_html_text = _html_text.replace("__PYTEST_SUMMARY__", _pytest_summary)
_html_text = _html_text.replace("__SYNTAX_SUMMARY__", _syntax_summary)
_html_text = _html_text.replace("__ROLLUP_DATA_JSON__", _data_json_safe)

DASHBOARD_HTML_PATH = ROLLUP_DIR / "bp4_executive_rollup_dashboard.html"
DASHBOARD_HTML_PATH.write_text(_html_text, encoding="utf-8")
print(
    f"[SAVED] {DASHBOARD_HTML_PATH.relative_to(PROJECT_ROOT)} ({DASHBOARD_HTML_PATH.stat().st_size:,} bytes)"
)

# ============================================================
# SECTION 8: Word document export (python-docx) - charts, tables, SMART suggestions, the new
# Recommended-for-Production status, and full Gate 1 / Gate 6 real-output detail (docx skill: US
# Letter page size set explicitly, Calibri professional font). No financial-impact section exists
# in this report.
# ============================================================
DOCX_PATH = ROLLUP_DIR / "bp4_executive_rollup_report.docx"
rollup.write_docx_report(BUNDLE, KPIS, SUGGESTIONS, FIGURES, TIER_DF, DOCX_PATH)
print(f"[SAVED] {DOCX_PATH.relative_to(PROJECT_ROOT)} ({DOCX_PATH.stat().st_size:,} bytes)")

# ============================================================
# SECTION 9: Excel workbook export (openpyxl) - 9 sheets (incl. 00_ReadMe), one per real
# gate-output area. No financial-impact sheet.
# ============================================================
XLSX_PATH = ROLLUP_DIR / "bp4_executive_rollup_workbook.xlsx"
rollup.write_xlsx_workbook(BUNDLE, KPIS, SUGGESTIONS, TIER_DF, XLSX_PATH)
print(f"[SAVED] {XLSX_PATH.relative_to(PROJECT_ROOT)} ({XLSX_PATH.stat().st_size:,} bytes)")

# ============================================================
# SECTION 10: PowerPoint deck export (python-pptx) - explicit slide size, RGBColor objects (never
# a literal '#' prefix or 8-digit hex), figures rendered once and reused (WARP). No financial
# slide.
# ============================================================
PPTX_PATH = ROLLUP_DIR / "bp4_executive_rollup_deck.pptx"
rollup.write_pptx_deck(BUNDLE, KPIS, SUGGESTIONS, FIGURES, TIER_DF, PPTX_PATH)
print(f"[SAVED] {PPTX_PATH.relative_to(PROJECT_ROOT)} ({PPTX_PATH.stat().st_size:,} bytes)")

# ============================================================
# SECTION 11: Executive-rollup manifest (audit trail - which real Gate 1-6 artifacts fed this
# report, and what was generated, when).
# ============================================================
from datetime import datetime, timezone  # noqa: E402

MANIFEST = {
    "report": "bp4_customer_journey_analytics_executive_rollup",
    "champion_pipeline": KPIS["champion_pipeline"],
    "recommended_for_production_tier": KPIS["production_recommendation"]["tier"],
    "source_artifacts_dir": str(BUNDLE["artifacts_dir"].relative_to(PROJECT_ROOT)),
    "source_artifact_files": sorted(p.name for p in BUNDLE["artifacts_dir"].glob("*") if p.is_file()),
    "outputs": {
        "dashboard_html": str(DASHBOARD_HTML_PATH.relative_to(PROJECT_ROOT)),
        "report_docx": str(DOCX_PATH.relative_to(PROJECT_ROOT)),
        "workbook_xlsx": str(XLSX_PATH.relative_to(PROJECT_ROOT)),
        "deck_pptx": str(PPTX_PATH.relative_to(PROJECT_ROOT)),
    },
    "output_sizes_bytes": {
        "dashboard_html": DASHBOARD_HTML_PATH.stat().st_size,
        "report_docx": DOCX_PATH.stat().st_size,
        "workbook_xlsx": XLSX_PATH.stat().st_size,
        "deck_pptx": PPTX_PATH.stat().st_size,
    },
    "contains_financial_impact_section": False,
    "contains_assumption_based_content": False,
    "n_gate3_slow_but_correct_candidates_represented": KPIS["n_gate3_slow_but_correct_candidates"],
    "n_high_and_medium_tier_clusters_embedded": len(_embed_clusters_df),
    "tier_2_reachable_for_this_bp": KPIS["production_recommendation"]["tier_2_reachable_for_this_bp"],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
MANIFEST_PATH = ARTIFACTS_DIR / "executive_rollup_manifest.json"
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2)
print(f"[SAVED] {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass.
# ============================================================
from docx import Document as _DocxDocument  # noqa: E402
from openpyxl import load_workbook as _load_workbook  # noqa: E402
from pptx import Presentation as _PptxPresentation  # noqa: E402

_docx_reopen_ok = _DocxDocument(str(DOCX_PATH)) is not None
_xlsx_reopen_ok = _load_workbook(str(XLSX_PATH)) is not None
_pptx_reopen_ok = _PptxPresentation(str(PPTX_PATH)) is not None

_saved_html = DASHBOARD_HTML_PATH.read_text(encoding="utf-8")
_embedded_json_text = (
    _saved_html.split('<script id="rollup-data" type="application/json">', 1)[1]
    .split("</script>", 1)[0]
    .replace("<\\/", "</")
)
_embedded_json = json.loads(_embedded_json_text)

_xlsx_wb_check = _load_workbook(str(XLSX_PATH))

_n_high_medium_real = int((BUNDLE["gate5_cluster_df"]["review_priority_tier"].isin(["HIGH", "MEDIUM"])).sum())

checks = {
    "dashboard_html_written_and_nonempty": DASHBOARD_HTML_PATH.exists()
    and DASHBOARD_HTML_PATH.stat().st_size > 10_000,
    "dashboard_html_embedded_json_parses": isinstance(_embedded_json, dict) and "kpis" in _embedded_json,
    "dashboard_html_includes_gate1_and_gate6": "gate1" in _embedded_json and "gate6_detail" in _embedded_json,
    "dashboard_html_includes_production_recommendation": "production_recommendation"
    in _embedded_json["kpis"],
    "dashboard_html_production_recommendation_tier_2_unreachable_stated": _embedded_json["kpis"][
        "production_recommendation"
    ]["tier_2_reachable_for_this_bp"]
    is False,
    "dashboard_html_clusters_embedded_match_real_high_medium_count": len(_embedded_json["clusters"])
    == _n_high_medium_real,
    "dashboard_html_tier_rollup_matches_gate5": len(_embedded_json["tier_rollup"]) == len(TIER_DF),
    "dashboard_html_monthly_trend_present": len(_embedded_json["monthly_trend"]) > 0,
    "dashboard_html_no_financial_content": "financial" not in _embedded_json
    and "Financial Impact" not in _saved_html,
    "dashboard_html_no_unresolved_placeholders": all(
        token not in _saved_html
        for token in (
            "__GENERATED_AT__",
            "__CHAMPION_PIPELINE__",
            "__PYTEST_SUMMARY__",
            "__SYNTAX_SUMMARY__",
            "__ROLLUP_DATA_JSON__",
        )
    ),
    "docx_written_and_nonempty": DOCX_PATH.exists() and DOCX_PATH.stat().st_size > 10_000,
    "xlsx_written_and_nonempty": XLSX_PATH.exists() and XLSX_PATH.stat().st_size > 5_000,
    "xlsx_has_gate1_and_gate6_sheets": (
        "02_Gate1_Journey_Definition" in _xlsx_wb_check.sheetnames
        and "07_Gate6_Governance" in _xlsx_wb_check.sheetnames
    ),
    "xlsx_has_all_9_sheets": len(_xlsx_wb_check.sheetnames) == 9,
    "xlsx_no_financial_sheet": not any("Financial" in name for name in _xlsx_wb_check.sheetnames),
    "xlsx_high_tier_sheet_matches_real_count": (
        _xlsx_wb_check["06_HIGH_Tier_Clusters"].max_row - 1
        == int((BUNDLE["gate5_cluster_df"]["review_priority_tier"] == "HIGH").sum())
    ),
    "pptx_written_and_nonempty": PPTX_PATH.exists() and PPTX_PATH.stat().st_size > 10_000,
    "manifest_written": MANIFEST_PATH.exists(),
    "docx_reopens_cleanly": _docx_reopen_ok,
    "xlsx_reopens_cleanly": _xlsx_reopen_ok,
    "pptx_reopens_cleanly": _pptx_reopen_ok,
    "smart_suggestions_all_grounded": len(SUGGESTIONS) >= 3,
    "tier_rollup_covers_all_four_tiers": len(TIER_DF) == 4,
    "production_recommendation_is_one_of_three_tiers": KPIS["production_recommendation"]["tier_code"]
    in (1, 2, 3),
    "production_recommendation_never_tier_2_for_bp4": KPIS["production_recommendation"]["tier_code"] != 2,
    "outputs_written_under_project_reports_folder": all(
        str(p).startswith(str(REPORTS_DIR)) for p in [DASHBOARD_HTML_PATH, DOCX_PATH, XLSX_PATH, PPTX_PATH]
    ),
}

print("\n=== INTEGRITY CHECKS ===")
for _name, _passed in checks.items():
    _status = "[PASS]" if _passed else "[FAIL]"
    print(f"{_status} {_name}")
    assert _passed, f"[CHECK FAILED] {_name}"

print(
    f"\n[ALL CHECKS PASSED] BP4 executive-rollup report complete. "
    f"Dashboard: {DASHBOARD_HTML_PATH.name} | Report: {DOCX_PATH.name} | "
    f"Workbook: {XLSX_PATH.name} | Deck: {PPTX_PATH.name}. "
    f"All outputs written under {ROLLUP_DIR.relative_to(PROJECT_ROOT)}. "
    "Every figure in every deliverable is a real, original BP4 Gate 1-6 notebook output - "
    "there is no financial-impact section and no assumption-based content anywhere in this report. "
    f"Recommended for Production status: {KPIS['production_recommendation']['tier']} "
    "(Tier 2 is structurally unreachable for BP4 - no disparate-impact check exists for this "
    "business problem)."
)
